In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

# ============================================================
# 1. SETTINGS
# ============================================================

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Fixed Autoencoder architecture
hidden_dim = 32
encoding_dim = 8
batch_size = 64
learning_rate = 1e-3
epochs = 100
patience = 10
validation_split = 0.2

train_fraction = 0.70

# Manual thresholds
threshold_list = [
    0.00006,
    0.0,
    0.0001
]

data_path = Path("../data_processed/NAB/industrial_machine_features_labeled.csv")
results_root = Path("../results/AE_NAB_thresholds")
results_root.mkdir(parents=True, exist_ok=True)

feature_cols = ["value", "diff1", "roll_mean", "roll_std", "roll_min", "roll_max"]
required_cols = feature_cols + ["y_true"]

# ============================================================
# 2. LOAD NAB DATASET
# ============================================================

df = pd.read_csv(data_path)

if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

df = df.dropna(subset=required_cols).reset_index(drop=True)

# ============================================================
# 3. CHRONOLOGICAL TRAIN/TEST SPLIT
# ============================================================

split_idx = int(len(df) * train_fraction)

train_window = df.iloc[:split_idx].copy()
test_window = df.iloc[split_idx:].copy()

X_train_normal_raw = train_window.loc[
    train_window["y_true"] == 0,
    feature_cols
].copy()

X_test_raw = test_window[feature_cols].copy()
y_test = test_window["y_true"].astype(int).copy()

if len(X_train_normal_raw) == 0:
    raise ValueError("The training window does not contain normal samples.")

print("Split type: chronological windows, no random sampling")
print("Train fraction:", train_fraction)
print("Total samples after preprocessing:", len(df))
print("Train window samples:", len(train_window))
print("Train normal samples:", len(X_train_normal_raw))
print("Test window samples:", len(test_window))
print("Test normal samples:", int((y_test == 0).sum()))
print("Test anomaly samples:", int((y_test == 1).sum()))

# ============================================================
# 4. SCALING
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_normal_raw)
X_test = scaler.transform(X_test_raw)

# ============================================================
# 5. TRAIN FIXED AUTOENCODER
# ============================================================

input_dim = X_train.shape[1]

def build_autoencoder(input_dim, hidden_dim, encoding_dim, learning_rate):
    inputs = Input(shape=(input_dim,))
    x = Dense(hidden_dim, activation="relu")(inputs)
    latent = Dense(encoding_dim, activation="relu")(x)
    x = Dense(hidden_dim, activation="relu")(latent)
    outputs = Dense(input_dim, activation="linear")(x)

    model = Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse"
    )
    return model

model = build_autoencoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    encoding_dim=encoding_dim,
    learning_rate=learning_rate
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=patience,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    X_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=validation_split,
    shuffle=True,
    callbacks=[early_stopping],
    verbose=0
)

train_recon = model.predict(X_train, verbose=0)
train_errors = np.mean(np.square(X_train - train_recon), axis=1)

test_recon = model.predict(X_test, verbose=0)
scores = np.mean(np.square(X_test - test_recon), axis=1)

print("Train reconstruction error min:", train_errors.min())
print("Train reconstruction error max:", train_errors.max())
print("Train reconstruction error mean:", train_errors.mean())

print("Test reconstruction error min:", scores.min())
print("Test reconstruction error max:", scores.max())
print("Test reconstruction error mean:", scores.mean())

print("Normal test error percentiles:")
print(np.percentile(scores[y_test.values == 0], [0, 25, 50, 75, 90, 95, 99, 100]))

print("Anomaly test error percentiles:")
print(np.percentile(scores[y_test.values == 1], [0, 25, 50, 75, 90, 95, 99, 100]))

# ============================================================
# 6. HELPERS
# ============================================================

def safe_name(value):
    return str(value).replace(".", "_").replace("-", "minus_")

# ============================================================
# 7. EXPERIMENTS WITH MANUAL THRESHOLDS
# ============================================================

all_results = []

for threshold in threshold_list:
    exp_name = (
        f"train_fraction_{safe_name(train_fraction)}_"
        f"hidden_{hidden_dim}_"
        f"latent_{encoding_dim}_"
        f"batch_{batch_size}_"
        f"lr_{safe_name(learning_rate)}_"
        f"threshold_{safe_name(threshold)}"
    )

    exp_dir = results_root / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)

    df_temp = test_window.copy()

    # 0 = normal, 1 = anomaly
    df_temp["reconstruction_error"] = scores
    df_temp["threshold"] = threshold
    df_temp["is_anomaly"] = (df_temp["reconstruction_error"] > threshold).astype(int)

    y_pred = df_temp["is_anomaly"].astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_test, y_pred) * 100
    precision = precision_score(y_test, y_pred, zero_division=0) * 100
    recall = recall_score(y_test, y_pred, zero_division=0) * 100
    f1 = f1_score(y_test, y_pred, zero_division=0) * 100

    result_row = {
        "experiment": exp_name,
        "train_fraction": train_fraction,
        "hidden_dim": hidden_dim,
        "encoding_dim": encoding_dim,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "patience": patience,
        "epochs_trained": len(history.history["loss"]),
        "best_val_loss": round(float(np.min(history.history["val_loss"])), 8),
        "final_train_loss": round(float(history.history["loss"][-1]), 8),
        "threshold": threshold,
        "split_type": "chronological_window_split",
        "train_window_samples": len(train_window),
        "train_normal_samples": len(X_train),
        "test_window_samples": len(X_test),
        "test_normal_samples": int((y_test == 0).sum()),
        "test_anomaly_samples": int((y_test == 1).sum()),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy_%": round(accuracy, 2),
        "precision_%": round(precision, 2),
        "recall_%": round(recall, 2),
        "f1_%": round(f1, 2)
    }

    all_results.append(result_row)

    df_temp.to_csv(exp_dir / "predictions.csv", index=False)

    print("======================================")
    print("Autoencoder NAB")
    print("======================================")
    print(f"Train fraction: {train_fraction}")
    print(f"Threshold: {threshold}")
    print(f"Accuracy : {accuracy:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"Recall   : {recall:.2f}%")
    print(f"F1-score : {f1:.2f}%")
    print("Confusion Matrix")
    print(cm)

    # ========================================================
    # Metrics table
    # ========================================================

    metrics_df = pd.DataFrame({
        "Metrika": [
            "Presnosť (Accuracy)",
            "Precíznosť (Precision)",
            "Citlivosť (Recall)",
            "F1-skóre"
        ],
        "Hodnota (%)": [
            round(accuracy, 2),
            round(precision, 2),
            round(recall, 2),
            round(f1, 2)
        ]
    })

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis("off")

    table = ax.table(
        cellText=metrics_df.values,
        colLabels=metrics_df.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("#dce6f1")
        else:
            cell.set_facecolor("#f9f9f9")

    plt.title(
        f"Výsledné metriky Autoencoder – NAB\n"
        f"Train fraction = {train_fraction}, Threshold = {threshold}",
        fontsize=12,
        pad=20
    )

    plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Confusion matrix
    # ========================================================

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Matica zámen – Threshold = {threshold}")
    plt.colorbar()

    classes = ["Normálne", "Anomálie"]
    tick_marks = np.arange(len(classes))

    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    threshold_cm = cm.max() / 2.0 if cm.max() > 0 else 0.5

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "black" if cm[i, j] > threshold_cm else "white"
            plt.text(
                j, i, cm[i, j],
                ha="center",
                va="center",
                color=color
            )

    plt.xlabel("Predikované triedy")
    plt.ylabel("Skutočné triedy")
    plt.tight_layout()
    plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Reconstruction error histogram
    # ========================================================
    
    normal_scores = scores[y_test.values == 0]
    anomaly_scores = scores[y_test.values == 1]
    
    plt.figure(figsize=(16, 6))
    
    bins = np.logspace(
        np.log10(max(scores.min(), 1e-8)),
        np.log10(scores.max()),
        80
    )
    
    plt.hist(
        normal_scores,
        bins=bins,
        alpha=0.7,
        label="Normálne"
    )
    
    plt.hist(
        anomaly_scores,
        bins=bins,
        alpha=0.7,
        label="Anomálie"
    )
    
    plt.axvline(
        threshold,
        linestyle="--",
        linewidth=2,
        label=f"Threshold = {threshold}"
    )
    
    plt.xscale("log")
    
    plt.title(
        f"Histogram rekonštrukčnej chyby Autoencoder – NAB\n"
        f"Train fraction = {train_fraction}, Threshold = {threshold}"
    )
    plt.xlabel("Rekonštrukčná chyba (log škála)")
    plt.ylabel("Počet vzoriek")
    plt.legend()
    plt.grid(True, which="both")
    plt.tight_layout()
    plt.savefig(exp_dir / "reconstruction_error_histogram.png", dpi=300, bbox_inches="tight")
    plt.close()

    # ========================================================
    # Anomaly plot over time
    # ========================================================

    if "timestamp" in df_temp.columns:
        x_axis = pd.to_datetime(df_temp["timestamp"])
        x_label = "Čas"
    else:
        x_axis = np.arange(len(df_temp))
        x_label = "Index"

    plt.figure(figsize=(14, 5))
    plt.plot(x_axis, df_temp["value"], linewidth=1, label="Teplota")

    anomaly_idx = df_temp["is_anomaly"] == 1

    plt.scatter(
        np.array(x_axis)[anomaly_idx],
        df_temp.loc[anomaly_idx, "value"],
        s=18,
        label="Detegované anomálie"
    )

    plt.title(
        f"Detekcia anomálií Autoencoder – NAB\n"
        f"Train fraction = {train_fraction}, Threshold = {threshold}"
    )
    plt.xlabel(x_label)
    plt.ylabel("Hodnota")
    plt.legend()
    plt.tight_layout()
    plt.savefig(exp_dir / "anomaly_plot.png", dpi=300, bbox_inches="tight")
    plt.close()

# ============================================================
# 8. SAVE HISTORY AND SUMMARY
# ============================================================

history_df = pd.DataFrame(history.history)
history_df.to_csv(results_root / "training_history.csv", index=False)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Split type: chronological windows, no random sampling
Train fraction: 0.7
Total samples after preprocessing: 22646
Train window samples: 15852
Train normal samples: 14718
Test window samples: 6794
Test normal samples: 5660
Test anomaly samples: 1134
Train reconstruction error min: 2.8519728046311613e-07
Train reconstruction error max: 0.001445352457718572
Train reconstruction error mean: 1.89331723031148e-05
Test reconstruction error min: 6.349366462372454e-07
Test reconstruction error max: 0.13328409665723231
Test reconstruction error mean: 0.00013763480135994434
Normal test error percentiles:
[6.34936646e-07 5.65585443e-06 1.20027849e-05 2.76255825e-05
 5.69777533e-05 9.42472295e-05 1.17080503e-03 1.33284097e-01]
Anomaly test error percentiles:
[4.43621636e-06 2.11678898e-05 4.85260210e-05 1.26165855e-04
 1.38093932e-04 1.43527616e-04 1.86278833e-03 1.08590213e-01]
Autoencoder NAB
Train fraction: 0.7
Threshold: 6e-05
Accuracy : 83.19%
Precision: 49.61%
Recall   : 44.71%
F1-score : 47